# RP-011: controlled two-epoch HNW pilot

Select **A100 40GB** or **L4 24GB** in Colab's runtime settings. Run cells in
order. Both arms warm-start from the same RP-010B best epoch-9 weights, use
fresh AdamW and no scheduler, and differ only in opt-in HNW. The fixed rich
prompt bank, 1,000 anchors, full training donor pool, seeds and loss settings
remain those in `docs/RP011_RUNBOOK.md`. Validation follows both epochs;
signed bias is **prediction minus GT** and differences are **HNW minus control**.

Preflight is mandatory. The separate pilot cell defaults to **RUN_PILOT=False**.
No test-set evaluation, additional experiment, resume, or RP-010 write occurs.
The source ZIP includes the current HNW patch; it does not rely on an older
GitHub revision containing these uncommitted changes.

## Drive inputs

Upload `RP011_source.zip` to `MyDrive/RP011_upload/` and open this notebook in
Colab. Reuse the existing asset archive and RP-010B folder:

```text
MyDrive/
├── RP011_upload/
│   ├── RP011_Colab.ipynb
│   └── RP011_source.zip
├── T2ICount-assets.zip                 # existing archive: pretrained + FSC147
└── T2ICount-assets/
    ├── prompts/fsc147_train1000_seed3407_gemma4_v3.json
    └── outputs/
        ├── rp010/seed3407/pair_v1/RP-010B/
        │   ├── best_epoch9.pth
        │   ├── manifest.json
        │   └── selection.json
        └── rp011/seed3407/pair_v1/      # created by this notebook; must be new
```

The existing asset ZIP may contain a `T2ICount-assets/` top folder or start
directly with `pretrained/` and `datasets/`. Only those two required input
subtrees are extracted onto the Colab SSD. An already expanded Drive
`T2ICount-assets/{pretrained,datasets/FSC147}` layout is also supported if
the ZIP is absent. No model weights, density files, or prompt bank are
included in the lightweight source ZIP. Reserve roughly 13GB of additional
Drive space for selected arm checkpoints and atomic checkpoint replacement.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shlex, shutil, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/T2ICount-assets')
SOURCE_ZIP = Path('/content/drive/MyDrive/RP011_upload/RP011_source.zip')
ASSET_ZIP = Path('/content/drive/MyDrive/T2ICount-assets.zip')
REPO = Path('/content/T2ICount-RP011')
ASSET_ROOT = Path('/content/T2ICount-assets')
DATA_ROOT = ASSET_ROOT / 'datasets/FSC147'
RP010_RUN = DRIVE_ROOT / 'outputs/rp010/seed3407/pair_v1/RP-010B'
CHECKPOINT = RP010_RUN / 'best_epoch9.pth'
PROMPT_BANK = DRIVE_ROOT / 'prompts/fsc147_train1000_seed3407_gemma4_v3.json'
PAIR_ID = 'pair_v1'  # use a new ID for every new/failed pilot attempt
assert PAIR_ID and all(c.isalnum() or c in '_-' for c in PAIR_ID)
OUTPUT_DIR = DRIVE_ROOT / 'outputs/rp011/seed3407' / PAIR_ID
assert RP010_RUN.resolve() not in OUTPUT_DIR.resolve().parents
assert OUTPUT_DIR.resolve() not in RP010_RUN.resolve().parents
assert not OUTPUT_DIR.exists(), f'Output exists: {OUTPUT_DIR}; preserve it and choose a new PAIR_ID'
NOTEBOOK_LOGS = OUTPUT_DIR / 'notebook'
NOTEBOOK_LOGS.mkdir(parents=True, exist_ok=False)
PREFLIGHT_PASSED = False

def stream(command, log_name):
    command = list(map(str, command))
    print('+', shlex.join(command), flush=True)
    with (NOTEBOOK_LOGS / log_name).open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'Command failed ({process.returncode}); inspect {log.name}')

print('Read-only source:', RP010_RUN)
print('New RP-011 outputs:', OUTPUT_DIR)


## Load the verified source snapshot

A local Git snapshot is created only under `/content/T2ICount-RP011` because
the existing runner records `git ls-files` and `git rev-parse HEAD` for its
provenance guard. Nothing is committed or pushed to the original repository.

In [ ]:
EXPECTED_SOURCE_MANIFEST_SHA256 = 'eb9f47e44703c507910ffa062b34cec967b94895f80940612d23aab2c42c0f95'
assert SOURCE_ZIP.is_file(), SOURCE_ZIP
assert not REPO.exists(), f'{REPO} already exists; use a fresh Colab runtime'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    raw = archive.read('SOURCE_MANIFEST.json')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED_SOURCE_MANIFEST_SHA256, 'Source ZIP manifest mismatch'
    source_manifest = json.loads(raw)
    assert source_manifest['schema'] == 'rp011-colab-upload-v1'
    assert set(archive.namelist()) == set(source_manifest['files']) | {'SOURCE_MANIFEST.json'}
    REPO.mkdir(parents=True, exist_ok=False)
    for name, expected in source_manifest['files'].items():
        target = (REPO / name).resolve()
        assert REPO.resolve() in target.parents, f'Unsafe source member: {name}'
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == expected, f'Source changed: {name}'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    (REPO / 'SOURCE_MANIFEST.json').write_bytes(raw)
for name in ('train.py', 'utils/regression_trainer.py', 'tools/train_rp011.py',
             'tools/train_rp010.py', 'configs/rp011.json', 'configs/rp010b.json'):
    assert (REPO / name).is_file(), name
stream(['git', 'init', '-q'], 'source.log')
stream(['git', '-c', 'core.autocrlf=false', 'add', '.'], 'source.log')
stream(['git', '-c', 'user.name=RP011 upload snapshot', '-c', 'user.email=rp011@localhost',
        'commit', '-qm', 'RP011 uploaded working-tree source snapshot'], 'source.log')
print('Original source HEAD:', source_manifest['origin_head'])
print('Current uploaded file hashes verified:', len(source_manifest['files']))


## Install the project's Colab compatibility bridge

Use `requirements-colab.txt`, not the old Python3.8/CUDA11.3 environment.
Existing Torch, Torchvision, Torchaudio and core numerical packages are
constrained to their installed versions; no `--upgrade` is used. The
taming-transformers and OpenAI CLIP Git revisions remain exactly pinned.
A resolver/import error stops the notebook; do not upgrade packages between
arms. A fresh subprocess checks imports so notebook-kernel module caching
cannot hide an installation problem.

In [ ]:
constraints = Path('/content/rp011-runtime-constraints.txt')
protected = ('torch', 'torchvision', 'torchaudio', 'numpy', 'Pillow', 'h5py',
             'opencv-python', 'opencv-python-headless')
installed = {}
for package in protected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
assert 'torch' in installed and 'torchvision' in installed, 'Select a standard Colab GPU runtime'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in installed.items()))
stream([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
        '-c', constraints, '-r', REPO / 'requirements-colab.txt'], 'dependencies.log')
for package, version in installed.items():
    assert importlib.metadata.version(package) == version, f'Protected package changed: {package}'
taming = REPO / 'src/taming-transformers'
assert taming.is_dir(), 'Pinned editable taming-transformers checkout missing'
os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO), str(taming), os.environ.get('PYTHONPATH', '')])
stream([sys.executable, '-c', 'import torch, torchvision, transformers, taming; '
        'from tools.train_rp011 import PilotTrainer; print("RP-011 imports OK")'], 'dependencies.log')
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], cwd=REPO, text=True)
(NOTEBOOK_LOGS / 'pip-freeze.txt').write_text(freeze)
shutil.copyfile(constraints, NOTEBOOK_LOGS / constraints.name)


## Check GPU and host RAM before loading the model

The existing RP-010B L4 smoke peaked at21.16GiB allocated. This initial check
rejects smaller GPUs; the paired preflight remains the authoritative RP-011
memory/gradient check. Prefer A10040GB for headroom or L424GB. Precision,
batch size, crop size and freezing are never changed to fit memory.

In [ ]:
gpu_probe = r'''
import json, torch
from pathlib import Path
from tools.train_rp010 import host_free_bytes, runtime_signature
assert torch.cuda.is_available(), 'CUDA unavailable: choose a GPU runtime'
assert torch.cuda.device_count() == 1, 'Use exactly one GPU'
free, total = torch.cuda.mem_get_info()
ram = host_free_bytes()
report = {'gpu': torch.cuda.get_device_name(0), 'free_bytes': free, 'total_bytes': total,
          'host_free_bytes': ram, 'runtime': runtime_signature()}
Path(__REPORT__).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
assert total >= 22 * 2**30, 'Need L4 24GB / A100 40GB class VRAM'
assert free >= 21 * 2**30, 'Insufficient free VRAM; do not change the experiment to fit'
assert ram is None or ram >= 8 * 2**30, 'Need at least 8 GiB free host RAM'
'''
gpu_probe = '\n'.join(line[4:] if line.startswith('    ') else line for line in gpu_probe.splitlines())
gpu_probe = gpu_probe.replace('__REPORT__', repr(str(NOTEBOOK_LOGS / 'gpu_runtime.json')))
stream([sys.executable, '-u', '-c', gpu_probe], 'gpu.log')


## Prepare existing dataset/pretrained inputs on the local SSD

Reuse `T2ICount-assets.zip` from the working project setup. Extract only
`pretrained/` and `datasets/FSC147/`. RP-010 outputs/checkpoints are not
extracted or modified. The warm-start model and prompt bank are read from
their existing Drive paths.

In [ ]:
required = [ASSET_ROOT / 'pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt',
            ASSET_ROOT / 'pretrained/clip-vit-large-patch14/config.json',
            DATA_ROOT / 'FSC_147/Train_Test_Val_FSC_147.json']
if not all(path.is_file() for path in required):
    ASSET_ROOT.mkdir(parents=True, exist_ok=True)
    if ASSET_ZIP.is_file():
        with zipfile.ZipFile(ASSET_ZIP) as archive:
            selected = []
            for info in archive.infolist():
                name = info.filename.replace('\\', '/')
                if name.startswith('T2ICount-assets/'):
                    name = name[len('T2ICount-assets/'):]
                if not name.startswith(('pretrained/', 'datasets/FSC147/')) or info.is_dir():
                    continue
                target = (ASSET_ROOT / name).resolve()
                assert ASSET_ROOT.resolve() in target.parents, f'Unsafe asset member: {name}'
                selected.append((info, target))
            assert selected, 'ZIP must contain pretrained/ and datasets/FSC147/'
            needed = sum(info.file_size for info, target in selected if not target.is_file())
            assert shutil.disk_usage('/content').free > needed + 3 * 2**30, 'Insufficient SSD space'
            for index, (info, target) in enumerate(selected):
                if target.is_file():
                    assert target.stat().st_size == info.file_size, f'Incomplete local asset: {target}; use fresh runtime'
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('xb') as destination:
                    shutil.copyfileobj(source, destination, length=8 * 1024**2)
                if index % 500 == 0:
                    print(f'Extracted {index + 1}/{len(selected)} input files', flush=True)
    else:
        for relative in ('pretrained', 'datasets/FSC147'):
            source, target = DRIVE_ROOT / relative, ASSET_ROOT / relative
            assert source.is_dir(), f'Missing {ASSET_ZIP} and expanded input directory {source}'
            assert not target.exists(), f'Partial local inputs at {target}; use a fresh runtime'
            shutil.copytree(source, target)
for path in required + [CHECKPOINT, PROMPT_BANK, RP010_RUN / 'manifest.json', RP010_RUN / 'selection.json']:
    assert path.is_file(), f'Missing required input: {path}'
print('Local FSC147:', DATA_ROOT)
print('Read-only warm start:', CHECKPOINT)


## Validate the actual checkpoint, recorded config and fixed data contract

Reuse the runner's `plan`, `verify_source` and `verify_data` functions.
Validation checks the exact checkpoint SHA, saved RP-010B configuration,
prompt-bank bytes, selected1,000 anchors, split/class/annotation hashes,
full donor-file coverage and validation coverage. As in the existing runner,
image/density payloads are assumed immutable; no historical per-payload byte
hashes exist. Strict model key/shape loading occurs in both GPU smokes.

In [ ]:
RUNNER = REPO / 'tools/train_rp011.py'
COMMON = ['--asset-root', str(ASSET_ROOT), '--rp010-run', str(RP010_RUN),
          '--output-dir', str(OUTPUT_DIR), '--data-dir', str(DATA_ROOT),
          '--prompt-bank', str(PROMPT_BANK)]
validation = r'''
import json
from pathlib import Path
from tools.train_rp011 import plan, verify_source, verify_data
spec, configs, run, output = plan(__ASSETS__, __SOURCE__, __OUTPUT__,
                                data_dir=__DATA__, prompt_bank=__BANK__)
manifest, selection = verify_source(spec, configs, run)
data = verify_data(configs['control'], manifest)
differences = [key for key in configs['control'] if configs['control'][key] != configs['hnw'][key]]
assert differences == ['rrcl_hnw']
result = {'selection': selection, 'effective_config_differences': differences, 'data': data}
Path(__REPORT__).write_text(json.dumps(result, indent=2))
print('Checkpoint SHA/config/data validation passed; 2000 updates per arm.')
'''
validation = '\n'.join(line[4:] if line.startswith('    ') else line for line in validation.splitlines())
for token, value in {'__ASSETS__': ASSET_ROOT, '__SOURCE__': RP010_RUN, '__OUTPUT__': OUTPUT_DIR,
                     '__DATA__': DATA_ROOT, '__BANK__': PROMPT_BANK,
                     '__REPORT__': NOTEBOOK_LOGS / 'input_validation.json'}.items():
    validation = validation.replace(token, repr(str(value)))
stream([sys.executable, '-u', '-c', validation], 'input_validation.log')


## Mandatory paired GPU preflight: HNW OFF and ON

This single supported command executes both existing smokes sequentially.
It strictly loads the model in both arms, compares starting-state hashes,
uses matched inputs and three rich-supervision forwards, checks the backward
gradients, and records GPU allocated/reserved peaks. It takes zero optimizer
steps. Any failure or pending result aborts here and leaves the pilot gated.

In [ ]:
PREFLIGHT_PASSED = False
stream([sys.executable, '-u', RUNNER, 'preflight', *COMMON], 'paired_preflight.log')
preflight = json.loads((OUTPUT_DIR / 'preflight/report.json').read_text())
assert preflight['status'] == 'passed', preflight.get('blockers', preflight.get('error'))
assert set(preflight['arms']) == {'control', 'hnw'}
assert all(item['strict_model_load'] for item in preflight['arms'].values())
assert preflight['arms']['control']['starting_weights_sha256'] == preflight['arms']['hnw']['starting_weights_sha256']
assert preflight['effective_config_differences'] == ['rrcl_hnw']
PREFLIGHT_PASSED = True
print('BOTH GPU SMOKES PASSED. Expected pilot updates: 2000/arm, 4000 total.')
for arm, report in preflight['arms'].items():
    print(arm, 'loss=', report['loss'], 'gradient L2=', report['gradient_l2'],
          'peak allocated GiB=', round(report['peak_allocated_bytes'] / 2**30, 2),
          'peak reserved GiB=', round(report['peak_reserved_bytes'] / 2**30, 2))
print('Estimated training GiB:', round(preflight['estimated_training_bytes'] / 2**30, 2))
print('Review the report before enabling the pilot cell.')


## Pilot: explicit enable, two existing arms only

Leave `RUN_PILOT=False` for preparation/preflight only. After reviewing a
successful paired preflight, change it to `True` and run this cell once.
The existing runner enforces the same source/data/runtime and warm-start
hashes again. Each arm starts fresh and validates all FSC147 validation
images after each epoch. Existing/partial arm directories are preserved;
use a new pair/output directory and repeat preflight after an interruption.

In [ ]:
RUN_PILOT = False
if RUN_PILOT:
    assert PREFLIGHT_PASSED, 'Both GPU smokes must pass in this notebook session'
    report = json.loads((OUTPUT_DIR / 'preflight/report.json').read_text())
    assert report['status'] == 'passed' and set(report['arms']) == {'control', 'hnw'}
    stream([sys.executable, '-u', RUNNER, 'train', *COMMON,
            '--arm', 'control', '--allow-training'], 'control_console.log')
    stream([sys.executable, '-u', RUNNER, 'train', *COMMON,
            '--arm', 'hnw', '--allow-training'], 'hnw_console.log')
    stream([sys.executable, '-u', RUNNER, 'compare', *COMMON], 'compare_console.log')
else:
    print('Pilot disabled. Set RUN_PILOT=True only after reviewing the passed preflight.')


## Persistent results

All logs and outputs are under the new Drive RP-011 directory:

```text
rp011/seed3407/<PAIR_ID>/
├── notebook/                    # setup, validation, preflight and arm console logs
├── preflight/report.json
├── control/                     # train.log, best.pth, val_epoch0/1.csv, records, completed.json
├── hnw/                         # same files
├── paired_validation.csv
└── paired_validation.json
```

These are raw measurements for researcher review. No scientific conclusion
is written to PROJECT_STATE or EXPERIMENT_LEDGER. Model weights are selected
using the unchanged validation MAE+RMSE rule. No RP-010 file is overwritten.

In [ ]:
comparison = OUTPUT_DIR / 'paired_validation.json'
if comparison.is_file():
    results = json.loads(comparison.read_text())
    print('Signed bias: prediction minus GT. Paired differences: HNW minus control.')
    for row in results['rows']:
        print(json.dumps(row, indent=2))
else:
    print('No completed pilot comparison yet. Outputs:', OUTPUT_DIR)
